## Load resume and posting

In [31]:
import pandas as pd
import numpy as np
import re

resume=pd.read_csv('Resume.csv')
postings=pd.read_csv('filtered_linkedin_postings.csv')

print(resume.shape,postings.shape)
print(resume['Category'].unique())
print(postings['matched_category'].unique())

(2484, 4) (2347, 10)
['HR' 'DESIGNER' 'INFORMATION-TECHNOLOGY' 'TEACHER' 'ADVOCATE'
 'BUSINESS-DEVELOPMENT' 'HEALTHCARE' 'FITNESS' 'AGRICULTURE' 'BPO' 'SALES'
 'CONSULTANT' 'DIGITAL-MEDIA' 'AUTOMOBILE' 'CHEF' 'FINANCE' 'APPAREL'
 'ENGINEERING' 'ACCOUNTANT' 'CONSTRUCTION' 'PUBLIC-RELATIONS' 'BANKING'
 'ARTS' 'AVIATION']
['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


## Confirm category names line up between the two datasets before pairing

In [32]:
resume_cats=set(resume['Category'].unique())
posting_cats=set(postings['matched_category'].unique())

print("In resumes but not postings:", resume_cats - posting_cats)
print("In postings but not resumes:", posting_cats - resume_cats)

In resumes but not postings: set()
In postings but not resumes: set()


In [33]:
try:
    print(type(clf), type(tfidf), type(le))
except NameError as e:
    print("Missing object:", e)

<class 'sklearn.linear_model._logistic.LogisticRegression'> <class 'sklearn.feature_extraction.text.TfidfVectorizer'> <class 'sklearn.preprocessing._label.LabelEncoder'>


In [ ]:
# Then in this notebook, reload them
import joblib
clf = joblib.load('step2_matching_model.pkl')
tfidf = joblib.load('step2_tfidf_vectorizer.pkl')
le = joblib.load('step2_label_encoder.pkl')

## Clean resume text

In [35]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

resume['resume_clean'] = resume['Resume_str'].apply(clean_text)

## Sample a manageable number of resumes per category

In [36]:
RESUMES_PER_CATEGORY =50

sampled_resumes = (
    resume
    .groupby('Category', group_keys=False)[resume.columns]
    .apply(lambda g: g.sample(min(len(g), RESUMES_PER_CATEGORY), random_state=42))
    .reset_index(drop=True)
)
print(sampled_resumes.shape)
print(sampled_resumes['Category'].value_counts())

(1158, 5)
Category
ACCOUNTANT                50
ADVOCATE                  50
AGRICULTURE               50
APPAREL                   50
ARTS                      50
AVIATION                  50
BANKING                   50
BUSINESS-DEVELOPMENT      50
INFORMATION-TECHNOLOGY    50
CHEF                      50
CONSTRUCTION              50
CONSULTANT                50
DIGITAL-MEDIA             50
DESIGNER                  50
ENGINEERING               50
FINANCE                   50
SALES                     50
FITNESS                   50
HEALTHCARE                50
HR                        50
TEACHER                   50
PUBLIC-RELATIONS          50
AUTOMOBILE                36
BPO                       22
Name: count, dtype: int64


## Pair each sampled resume with postings from its OWN category

In [ ]:


pairs = []
categories = sampled_resumes['Category'].unique()

for _, resume_row in sampled_resumes.iterrows():
    cat = resume_row['Category']

    same_cat_postings = postings[postings['matched_category'] == cat].sample(
        min(2, len(postings[postings['matched_category'] == cat]))
    )
    for _, post_row in same_cat_postings.iterrows():
        pairs.append({
            'resume_id': resume_row['ID'],
            'resume_category': cat,
            'resume_clean': resume_row['resume_clean'],
            'job_id': post_row['job_id'],
            'posting_title': post_row['title'],
            'posting_category': post_row['matched_category'],
            'jd_clean': post_row['jd_clean'],
            'pair_type': 'same_category'
        })

    other_cats = [c for c in categories if c != cat]
    random_other_cat = random.choice(other_cats)
    diff_cat_posting = postings[postings['matched_category'] == random_other_cat].sample(1)
    for _, post_row in diff_cat_posting.iterrows():
        pairs.append({
            'resume_id': resume_row['ID'],
            'resume_category': cat,
            'resume_clean': resume_row['resume_clean'],
            'job_id': post_row['job_id'],
            'posting_title': post_row['title'],
            'posting_category': post_row['matched_category'],
            'jd_clean': post_row['jd_clean'],
            'pair_type': 'different_category'
        })

case_study_df = pd.DataFrame(pairs)
print(case_study_df.shape)
print(case_study_df['pair_type'].value_counts())

(3474, 5000)


## Build the SAME feature representation your Step 2

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def build_features(df, tfidf):
    resume_vecs = tfidf.transform(df['resume_clean'])
    jd_vecs = tfidf.transform(df['jd_clean'])
    sims = np.array([
        cosine_similarity(resume_vecs[i], jd_vecs[i])[0, 0]
        for i in range(df.shape[0])
    ])
    resume_len = df['resume_clean'].str.split().str.len().values
    jd_len = df['jd_clean'].str.split().str.len().values
    return np.column_stack([sims, resume_len, jd_len])

X_case_study = build_features(case_study_df, tfidf)  # `tfidf` must be the SAME fitted vectorizer from Step 2
print(X_case_study.shape)

## Run the trained matching model on these new pairs

In [ ]:
y_pred_case = clf.predict(X_case_study)          # `clf` = your Step 2 Logistic Regression, macro-F1 0.392
y_proba_case = clf.predict_proba(X_case_study)

case_study_df['predicted_fit'] = le.inverse_transform(y_pred_case)  # `le` = Step 2's LabelEncoder
case_study_df['fit_confidence'] = y_proba_case.max(axis=1)

case_study_df[['resume_category', 'posting_title', 'posting_category', 'pair_type', 'predicted_fit', 'fit_confidence']].head(20)

,resume_category,posting_title,posting_category,pair_type,predicted_fit,fit_confidence
0,ACCOUNTANT,Director of Accounting,ACCOUNTANT,same_category,Potential Fit,1.000000
1,ACCOUNTANT,Staff Accountant,ACCOUNTANT,same_category,Potential Fit,1.000000
2,ACCOUNTANT,Human Resources Assistant,HR,different_category,Potential Fit,0.903003
3,ACCOUNTANT,Senior Accountant - HQ of Large Manufacturer O...,ACCOUNTANT,same_category,No Fit,0.774493
4,ACCOUNTANT,Senior Accountant - BSC (Hybrid),ACCOUNTANT,same_category,No Fit,0.999519
5,ACCOUNTANT,Human Resources Director,HR,different_category,No Fit,0.999864
6,ACCOUNTANT,Senior Accountant,ACCOUNTANT,same_category,Potential Fit,0.879122
7,ACCOUNTANT,Staff Accountant,ACCOUNTANT,same_category,Potential Fit,0.999851
8,ACCOUNTANT,Outside Sales Representative,SALES,different_category,Potential Fit,0.999973
9,ACCOUNTANT,Senior Accountant - BSC (Hybrid),ACCOUNTANT,same_category,No Fit,0.960916


## Sanity check: does predicted fit align with expectation?

In [40]:
print(pd.crosstab(case_study_df['pair_type'], case_study_df['predicted_fit']))

predicted_fit       Good Fit  No Fit  Potential Fit
pair_type                                          
different_category        32     929            197
same_category             80    1746            490


## Save results

In [41]:
case_study_df.to_csv('case_study_results.csv', index=False)
print(f"Saved {len(case_study_df)} case study pairs")

Saved 3474 case study pairs
